In [ ]:
import mysql.connector
import pandas as pd
import json
import requests
import random
import datetime
import times
import math

In [ ]:
#Connection to MySQL server, parameters must be filled with correct information otherwise it won't work
conn = mysql.connector.connect(
    host = 'must_be_inserted',
    user = 'must_be_inserted',
    password = 'must_be_inserted',
    database = 'test'
)
cursor = conn.cursor()

In [ ]:
#Representing the webpage url, where only u will be changing in iterations
BASE_URL = "https://bitcointalk.org/index.php?action=profile;u={}" 

HEADERS_LIST = [
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/118.0.5993.118 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:118.0) "
                      "Gecko/20100101 Firefox/118.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Edge/118.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; Trident/7.0; rv:11.0) like Gecko",
        "Accept": "text/html, application/xhtml+xml, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    },
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/112.0.5615.138 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive"
    }
]

In [ ]:
# Define the file path for the proxy list, also must be inserted
path = r"path_to_ip_addresses_must_be_inserted"
proxies_dict_list = []

# Open the text file containing the proxies in read mode with UTF-8 encoding
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # split the TXT format IP:PORT:USER:PASS
        ip, port, user, pwd = line.split(":")
        
        # create a dict directly for requests
        proxy_dict = {
            "http": f"http://{user}:{pwd}@{ip}:{port}",
            "https": f"http://{user}:{pwd}@{ip}:{port}"
        }

        proxies_dict_list.append(proxy_dict)

# Functions

In [ ]:
def html_request(addr,a, offset=0):
    """
    Extracting transaction data for inserted address from blockchain explorer.
    """
    url = f"https://blockchain.info/rawaddr/{addr}"
    params = {
        "offset" : offset
    }
    test = True
    j = 0
    while test:
        j += 1
        try:
            #Parameter timeout: the first number is the waiting time for connection, the second number is the waiting time to generate response, obtain data from the webpage
            r = requests.get(url, proxies = proxies_dict_list[a], headers=HEADERS_LIST[0], timeout = (10,90), params = params)
            r = r.json()
            r["n_tx"]
            test = False
        except Exception as e:
            print(f"{j} error {addr}")
            print(e)
            time.sleep(5)
            a += 1

            # the number in if condition is equivalent to the number of IP addresses
            if a >= 250:
                a = 0
    return r, a

In [ ]:
def blockchain_data_data (r,l):
    """
    Extracting columns for blockchain_data table in MySQL (txid, num_inputs, num_outputs, fee, mempool_entry_time, block_height).
    There are 2 inputs r, which represents the transaction information for specific address and l, which represents the order number of transaction in the transaction history
    Output is one row with the same columns as blockchain_data table.
    """
    #blockchain_data
    txid = r["txs"][l]["hash"]
    num_inputs = r["txs"][l]["vin_sz"]
    num_outputs = r["txs"][l]["vout_sz"]
    fee = r["txs"][l]["fee"]/r["txs"][l]["size"]
    mempool_entry_time = r["txs"][l]["time"]
    mempool_entry_time = datetime.datetime.fromtimestamp(mempool_entry_time).strftime("%Y-%m-%d %H:%M:%S")
    block_height = r["txs"][l]["block_height"]
    row2 = [txid, num_inputs, num_outputs, fee, mempool_entry_time, block_height]
    return row2

def tx_inputs_data (r,l):
    """
    Extracting columns for tx_inputs table in MySQL (txid, input_order, address, address, value).
    There are 2 inputs r, which represents the transaction information for specific address and l, which represents the number of transaction in the transaction history
    Output is batch of data with the same columns as tx_inputs table for each input in specific transaction with order number l.
    """
    #tx_inputs
    batch = []
    txid = r["txs"][l]["hash"]
    num_inputs = r["txs"][l]["vin_sz"]
    for j in range(num_inputs):
        try:
            input_order = j
            address = r["txs"][l]["inputs"][j]["prev_out"]["addr"]
            #Value converted from BTC to sats
            value = r["txs"][l]["inputs"][j]["prev_out"]["value"]/100000000 
            row3 = [txid, input_order, address, value]
            batch.append(row3)
        except KeyError as e:
            print("Chyba")
    return batch

def tx_outputs_data (r,l):
    """
    Extracting columns for tx_outputs table in MySQL (txid, output_order, address, value).
    There are 2 inputs r, which represents the transaction information for specific address and l, which represents the number of transaction in the transaction history
    Output is batch of data with the same columns as tx_outputs table for each output in specific transaction with order number l.
    """
    #tx_outputs
    batch = []
    txid = r["txs"][l]["hash"]
    num_outputs = r["txs"][l]["vout_sz"]
    for k in range(num_outputs):

        try:
            output_order = k
            out = r["txs"][l]["out"][k]
            address = out["addr"]
            #Value converted from BTC to sats
            value = out["value"]/100000000
            row4 = [txid, output_order, address, value]
            batch.append(row4)
        except KeyError as e:
            print("Chyba")
    return batch

In [ ]:
def address_write (addr,a,ccc):
    """
    Writing blockchain data for specific BTC address into MySQL database and updating scraped_data table with the written data 
    (number of transactions in written address, last transaction, and if the data were written into SQL the format is changed to Valid)
    """
    batch_mempool_log = []
    batch_blockchain_data = []
    batch_tx_inputs = []
    batch_tx_outputs = []

    if a >= 250:
        a = 0
    r, a = html_request(addr,a)
    if r["n_tx"] > 0:
        ts = r["txs"][0]["time"]
        ts = datetime.datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M:%S")
        data_addr.iloc[ccc,4] = r["n_tx"]
        data_addr.iloc[ccc,5] = ts
        if r["n_tx"] > 100:
            iteracie = 100
        else:
            iteracie = r["n_tx"]
            
        for l in range(iteracie):
            row2 = blockchain_data_data(r,l)
            batch_blockchain_data.append(row2)
            
            batch = tx_inputs_data(r,l)
            batch_tx_inputs += batch
                
            batch = tx_outputs_data(r,l)
            batch_tx_outputs += batch
        
        if (r["n_tx"]) > 100:
            dodatocne_iteracie = math.floor(r["n_tx"]/100)
            if (r["n_tx"]) > 10000:
                print(f"Address {addr} with {r["n_tx"]} transactions")
            offset = 100
            iteracie = 100
            for m in range(dodatocne_iteracie):
                if (m+1) == dodatocne_iteracie:
                    iteracie = r["n_tx"]%100

                a += 1
                if a >= 250:
                    a = 0
                r, a = html_request(addr,a, offset)
                        
                for l in range(iteracie):
                    row2 = blockchain_data_data(r,l)
                    batch_blockchain_data.append(row2)
                    
                    batch = tx_inputs_data(r,l)
                    batch_tx_inputs += batch
                        
                    batch = tx_outputs_data(r,l)
                    batch_tx_outputs += batch
                offset += 100
                
        chunk_size = 1000

        if batch_blockchain_data:
            chunk = batch_blockchain_data[:chunk_size]
            sql = "INSERT IGNORE INTO blockchain_data (txid, num_inputs, num_outputs, fee, mempool_entry_time, block_height) VALUES (%s, %s, %s, %s, %s, %s)"
            cursor.executemany(sql, chunk)
            batch_blockchain_data = batch_blockchain_data[chunk_size:]
            conn.commit()

        
        if batch_tx_inputs:
            chunk = batch_tx_inputs[:chunk_size]
            sql = "INSERT IGNORE INTO tx_inputs (txid, input_order, address, value) VALUES (%s, %s, %s, %s)"
            cursor.executemany(sql, chunk)
            batch_tx_inputs = batch_tx_inputs[chunk_size:]
            conn.commit()

        if batch_tx_outputs:
            chunk = batch_tx_outputs[:chunk_size]
            sql = "INSERT IGNORE INTO tx_outputs (txid, output_order, address, value) VALUES (%s, %s, %s, %s)"
            cursor.executemany(sql, chunk)
            batch_tx_outputs = batch_tx_outputs[chunk_size:]
            conn.commit()
        
        
        batch_blockchain_data = []
        batch_tx_inputs = []
        batch_tx_outputs = []  
        data_addr.iloc[ccc,3] = "Valid Format"
    return a

# Main Code

In [ ]:
#MySQL code can be changed based on for which addresses we want to write transaction history into MySQL database
cursor.execute("SELECT id, address, location, btc_format, num_txs, last_txs FROM scraped_data WHERE address is not null order by id;")
data = cursor.fetchall()
data_addr = pd.DataFrame(data, columns = ["id", "address", "location", "btc_format", "num_txs", "last_txs"])
data_addr.head()

In [ ]:
#Writing transaction data for each address from data_addr dataframe
a = 0
start = 0
end = len(data_addr)
for i in range(start, end):
    address = data_addr.iloc[i,1]
    a = address_write(address,a,i)
    a += 1
    if a>= 250:
        a = 0
    print(i)